# Importation of Librariries

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import glob
import numpy as np
import json
import random
from datetime import datetime, timedelta
import datetime
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
import joblib

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Importation of Data

In [ ]:
# Set the path to the directory containing your folders
path = "Clips"

In [ ]:
# Get a list of all the folders in the directory
folder_list = os.listdir(path)

In [ ]:
# Create a DataFrame with one column that contains the folder names
df = pd.DataFrame({'Folder Name': folder_list})

In [ ]:
# Display the DataFrame
print(df)

In [ ]:
df

# Data Preprocessing

In [ ]:
# Filter the DataFrame to keep only the folders that start with 'W00'
df_filtered = df[df['Folder Name'].str.startswith('W00')]

In [ ]:
# Display the filtered DataFrame
print(df_filtered)

In [ ]:
df_filtered

In [ ]:
# Reset the index of the filtered DataFrame
df_filtered = df_filtered.reset_index(drop=True)

In [ ]:
df_filtered

In [ ]:
# Add a new column to indicate if each folder contains a JSON file
df_filtered['Contains JSON'] = [bool(glob.glob(os.path.join(path, folder, '*.json'))) for folder in df_filtered['Folder Name']]

In [ ]:
df_filtered

In [ ]:
# Count the number of folders with and without JSON files
count_json = df_filtered['Contains JSON'].value_counts()

In [ ]:
# Create a bar chart to show the counts
plt.bar(count_json.index.astype(str), count_json.values, color=['grey', 'blue'])
plt.title("Number of Folders with and without JSON files")
plt.xlabel("Contains JSON")
plt.ylabel("Count")
plt.show()

In [ ]:
# Get the count of folders with JSON files and without JSON files
count_with_json = count_json[True] if True in count_json.index else 0
count_without_json = count_json[False] if False in count_json.index else 0

# Print the counts
print(f"Folders with JSON files: {count_with_json}")
print(f"Folders without JSON files: {count_without_json}")

In [ ]:
# Filter the DataFrame to keep only the rows with a JSON file
df_with_json = df_filtered[df_filtered['Contains JSON']]

# Reset the index of the filtered DataFrame
df_with_json = df_with_json.reset_index(drop=True)

In [ ]:
# Print the new DataFrame
print(df_with_json)

In [ ]:
df_with_json

In [ ]:
# Drop the "Contains JSON" column
df_with_json = df_with_json.drop("Contains JSON", axis=1)

In [ ]:
df_with_json

In [ ]:
# Add the new columns with null values
df_with_json = df_with_json.assign(
    Deviation=np.nan,
    Maneuver=np.nan,
    WeatherCondition=np.nan,
    ParkingGarage=np.nan,
    DrivingMode=np.nan,
    ArtificialLight=np.nan,
    DynamicObjects=np.nan,
    DirectionChange=np.nan,
    LightCondition=np.nan,
    ParkingManeuver=np.nan,
    Seasons=np.nan,
    StaticObjects=np.nan,
    Location=np.nan
)

In [ ]:
# Print the new DataFrame
print(df_with_json)

In [ ]:
df_with_json

In [ ]:
# Rename the 'Folder Name' column to 'Clips'
df_with_json = df_with_json.rename(columns={'Folder Name': 'Clip'})

In [ ]:
df_with_json

In [ ]:
features = ['Deviation', 'Maneuver', 'WeatherCondition', 'ParkingGarage', 'DrivingMode', 'ArtificialLight', 'DynamicObjects', 'DirectionChange', 'LightCondition', 'ParkingManeuver', 'Seasons', 'StaticObjects', 'Location']

for feature in features:
    for folder in df_with_json['Clip']:
        folder_path = os.path.join(path, folder)
        json_file = [f for f in os.listdir(folder_path) if f.endswith('.json')][0]
        json_path = os.path.join(folder_path, json_file)
        with open(json_path, 'r') as f:
            try:
                data = json.load(f)
                if 'prelabels' in data and any(d['category'] == feature for d in data['prelabels']):
                    feature_label = [d['label'] for d in data['prelabels'] if d['category'] == feature][0]
                    df_with_json.loc[df_with_json['Clip'] == folder, feature] = feature_label
            except (FileNotFoundError, json.JSONDecodeError):
                continue


In [ ]:
df_with_json

In [ ]:
df_with_json

In [ ]:
Max_Nan= 6
df_with_json = df_with_json.dropna(thresh=len(df_with_json.columns) - Max_Nan)

In [ ]:
df_with_json = df_with_json.reset_index(drop=True)
df_with_json

In [ ]:
# Add the new columns with null values
df_with_json = df_with_json.assign(
    VID_START_FRAME=np.nan,
    VID_START_ELV=np.nan,
    VID_START_LAT=np.nan,
    VID_START_LNG=np.nan,
    VID_START_TSTMP=np.nan,
    SCN_START_FRAME=np.nan,
    SCN_START_ELV=np.nan,
    SCN_START_LAT=np.nan,
    SCN_START_LNG=np.nan,
    SCN_START_TSTMP=np.nan,
    VID_END_FRAME=np.nan,
    VID_END_ELV=np.nan,
    VID_END_LAT=np.nan,
    VID_END_LNG=np.nan,
    VID_END_TSTMP=np.nan,
    SCN_END_FRAME=np.nan,
    SCN_END_ELV=np.nan,
    SCN_END_LAT=np.nan,
    SCN_END_LNG=np.nan,
    SCN_END_TSTMP=np.nan,
)

In [ ]:
df_with_json

In [ ]:
for i, folder in enumerate(df_with_json['Clip']):
    folder_path = os.path.join(path, folder)
    json_file = [f for f in os.listdir(folder_path) if f.endswith('.json')][0]
    json_path = os.path.join(folder_path, json_file)
    with open(json_path, 'r') as f:
        try:
            data = json.load(f)
            if 'prelabels' in data:
                for prelabel in data['prelabels']:
                    if prelabel['category'] == 'Deviation':
                        state = prelabel['state']
                        prefix = 'VID_START_' if state == 'activated' else 'VID_END_'
                        df_with_json.loc[i, prefix + 'FRAME'] = prelabel['frame_number']
                        df_with_json.loc[i, prefix + 'LAT'] = prelabel['postion_coordinates']['latitude']
                        df_with_json.loc[i, prefix + 'LNG'] = prelabel['postion_coordinates']['longitude']
                        df_with_json.loc[i, prefix + 'ELV'] = prelabel['postion_coordinates']['elevation']
                        df_with_json.loc[i, prefix + 'TSTMP'] = prelabel['timestamp']
        except (FileNotFoundError, json.JSONDecodeError):
            continue

In [ ]:
df_with_json

In [ ]:
for folder in df_with_json['Clip']:
    folder_path = os.path.join(path, folder)
    json_file = [f for f in os.listdir(folder_path) if f.endswith('.json')][0]
    json_path = os.path.join(folder_path, json_file)
    with open(json_path, 'r') as f:
        try:
            data = json.load(f)
            if 'prelabels' in data:
                prelabels = data['prelabels']
                for prelabel in prelabels:
                    if 'label' in prelabel and prelabel['label'] == 'SCN_1_START':
                        timestamp = prelabel['timestamp']
                        frame_number = prelabel['frame_number']
                        latitude = prelabel['postion_coordinates']['latitude']
                        longitude = prelabel['postion_coordinates']['longitude']
                        elevation = prelabel['postion_coordinates']['elevation']
                        df_with_json.loc[df_with_json['Clip'] == folder, 'SCN_START_TSTMP'] = timestamp
                        df_with_json.loc[df_with_json['Clip'] == folder, 'SCN_START_FRAME'] = frame_number
                        df_with_json.loc[df_with_json['Clip'] == folder, 'SCN_START_LAT'] = latitude
                        df_with_json.loc[df_with_json['Clip'] == folder, 'SCN_START_LNG'] = longitude
                        df_with_json.loc[df_with_json['Clip'] == folder, 'SCN_START_ELV'] = elevation
                        break  # Exit the loop after finding the first matching prelabel
        except (FileNotFoundError, json.JSONDecodeError):
            continue

In [ ]:
df_with_json

In [ ]:
for folder in df_with_json['Clip']:
    folder_path = os.path.join(path, folder)
    json_file = [f for f in os.listdir(folder_path) if f.endswith('.json')][0]
    json_path = os.path.join(folder_path, json_file)
    with open(json_path, 'r') as f:
        try:
            data = json.load(f)
            if 'prelabels' in data:
                prelabels = data['prelabels']
                for prelabel in prelabels:
                    if 'label' in prelabel and prelabel['label'] == 'SCN_2_STOP':
                        timestamp = prelabel['timestamp']
                        frame_number = prelabel['frame_number']
                        latitude = prelabel['postion_coordinates']['latitude']
                        longitude = prelabel['postion_coordinates']['longitude']
                        elevation = prelabel['postion_coordinates']['elevation']
                        df_with_json.loc[df_with_json['Clip'] == folder, 'SCN_END_TSTMP'] = timestamp
                        df_with_json.loc[df_with_json['Clip'] == folder, 'SCN_END_FRAME'] = frame_number
                        df_with_json.loc[df_with_json['Clip'] == folder, 'SCN_END_LAT'] = latitude
                        df_with_json.loc[df_with_json['Clip'] == folder, 'SCN_END_LNG'] = longitude
                        df_with_json.loc[df_with_json['Clip'] == folder, 'SCN_END_ELV'] = elevation
                        break  # Exit the loop after finding the first matching prelabel
        except (FileNotFoundError, json.JSONDecodeError):
            continue

In [ ]:
df_with_json

In [ ]:
df_with_json.head(300).to_csv('flask_data.csv', index=False)

In [ ]:
# Define the feature sets
feature_sets = [
    ['VID_START_TSTMP', 'SCN_START_TSTMP', 'SCN_END_TSTMP', 'VID_END_TSTMP'],
    ['VID_START_FRAME', 'SCN_START_FRAME', 'SCN_END_FRAME', 'VID_END_FRAME']
]

# Create sub-dataframes for each feature set
sub_dataframes = []
for features in feature_sets:
    sub_dataframe = df_with_json[features].copy()
    sub_dataframe['Clip ID'] = df_with_json['Clip']
    sub_dataframes.append(sub_dataframe)

# Display sub-dataframes
timestamp_df = pd.concat(sub_dataframes[0:1], axis=1)
frame_number_df = pd.concat(sub_dataframes[1:2], axis=1)

print("Timestamp DataFrame:")
print(timestamp_df)
print()

print("Frame Number DataFrame:")
print(frame_number_df)
print()

In [ ]:
timestamp_df

In [ ]:
frame_number_df

# Data Modelling

In [ ]:
# Set the initial value of the VALIDITY_SCN_START column as "Invalid"
df_with_json['VALIDITY_SCN_START'] = 'Invalid'

# Update the VALIDITY_SCN_START column based on the conditions
df_with_json.loc[(df_with_json['VID_START_TSTMP'].notnull()) & (df_with_json['SCN_START_TSTMP'].notnull()), 'VALIDITY_SCN_START'] = \
    df_with_json.loc[(df_with_json['VID_START_TSTMP'].notnull()) & (df_with_json['SCN_START_TSTMP'].notnull())].apply(
        lambda row: 'Valid' if (row['SCN_START_TSTMP'] - row['VID_START_TSTMP']) < 30 * 1e9 else 'Invalid', axis=1)



In [ ]:
df_with_json

In [ ]:
# Set the initial value of the VALIDITY_SCN_START column as "Invalid"
df_with_json['VALIDITY_SCN_END'] = 'Invalid'

# Update the VALIDITY_SCN_START column based on the conditions
df_with_json.loc[(df_with_json['VID_END_TSTMP'].notnull()) & (df_with_json['SCN_END_TSTMP'].notnull()), 'VALIDITY_SCN_END'] = \
    df_with_json.loc[(df_with_json['VID_END_TSTMP'].notnull()) & (df_with_json['SCN_END_TSTMP'].notnull())].apply(
        lambda row: 'Valid' if (row['VID_END_TSTMP'] - row['SCN_END_TSTMP']) < 10 * 1e9 else 'Invalid', axis=1)


In [ ]:
df_with_json

In [ ]:
# Count the occurrences of each validity value
validity_counts_start = df_with_json['VALIDITY_SCN_START'].value_counts()
validity_counts_end = df_with_json['VALIDITY_SCN_END'].value_counts()

# Create separate bar plots for Scene Start Validity and Scene End Validity
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6))

# Plot for Scene Start Validity
ax1.bar(validity_counts_start.index, validity_counts_start.values, color='blue', alpha=0.5)
ax1.set_xlabel('Validity')
ax1.set_ylabel('Count')
ax1.set_title('Validity of Scene Start')

# Plot for Scene End Validity
ax2.bar(validity_counts_end.index, validity_counts_end.values, color='green', alpha=0.5)
ax2.set_xlabel('Validity')
ax2.set_ylabel('Count')
ax2.set_title('Validity of Scene End')

# Adjust spacing between subplots
plt.tight_layout()

# Show the plots
plt.show()

In [ ]:
# Create the CLIP_VALIDITY column
df_with_json['CLIP_VALIDITY'] = np.where(
    (df_with_json['VALIDITY_SCN_START'] == 'Invalid') | (df_with_json['VALIDITY_SCN_END'] == 'Invalid'),
    'Invalid',
    'Valid'
)

In [ ]:
df_with_json

In [ ]:
# Count the occurrences of each value in the CLIP_VALIDITY column
validity_counts = df_with_json['CLIP_VALIDITY'].value_counts()

# Create a bar plot
plt.bar(validity_counts.index, validity_counts.values)

# Set the plot title and labels
plt.title('Clip Validity')
plt.xlabel('Validity')
plt.ylabel('Count')

# Show the plot
plt.show()

In [ ]:
df_with_json = df_with_json.drop(['VALIDITY_SCN_START', 'VALIDITY_SCN_END', 'CLIP_VALIDITY'], axis=1)


In [ ]:
# Exclude numeric columns and 'Clip'
non_numeric_columns = df_with_json.select_dtypes(exclude=[np.number]).drop(columns=['Clip']).columns

# Iterate over non-numeric columns and print unique values
for column in non_numeric_columns:
    unique_values = df_with_json[column].unique()
    print(f"Unique values for column '{column}': {unique_values}")

In [ ]:
df_with_json

# Data Augmetation

In [ ]:
import random
import string

# List of non-numeric columns (excluding 'Clip')
non_numeric_columns = [col for col in df_with_json.columns if col != 'Clip' and not df_with_json[col].dtype.kind in 'biufc']

# Create a new empty dataframe for augmented data
augmented_df = pd.DataFrame(columns=df_with_json.columns)

# Iterate for 10 times for data augmentation
for _ in range(10):
    # Create a temporary dataframe to store the augmented data
    temp_df = pd.DataFrame(columns=df_with_json.columns)

    # Iterate over each row in the original dataframe
    for i, row in df_with_json.iterrows():
        # Create a new row for the augmented data
        augmented_row = row.copy()

        # Generate random values for non-numeric columns
        for column in non_numeric_columns:
            possible_values = df_with_json[column].dropna().unique()  # Get unique non-null values for the column
            random_value = random.choice(possible_values)  # Randomly select a value from possible values
            augmented_row[column] = random_value

        # Append the augmented row to the temporary dataframe
        temp_df = temp_df.append(augmented_row, ignore_index=True)

    # Generate random Clip IDs for the temporary dataframe
    temp_df['Clip'] = ['DA' + ''.join(random.choices(string.ascii_uppercase, k=5)) for _ in range(len(temp_df))]

    # Concatenate the augmented data to the final augmented dataframe
    augmented_df = pd.concat([augmented_df, temp_df], ignore_index=True)

# Display the augmented dataframe
print(augmented_df)


In [ ]:
augmented_df

In [ ]:
# List of numerical columns
numerical_columns = [col for col in augmented_df.columns if augmented_df[col].dtype.kind in 'biufc']

# Set numerical columns as NaN
augmented_df[numerical_columns] = float('NaN')

In [ ]:
augmented_df

In [ ]:
augmented_df.columns

In [ ]:
# Specify the numerical columns
numerical_columns = ['VID_START_FRAME', 'VID_START_ELV', 'VID_START_LAT', 'VID_START_LNG', 'VID_START_TSTMP',
                     'SCN_START_FRAME', 'SCN_START_ELV', 'SCN_START_LAT', 'SCN_START_LNG', 'SCN_START_TSTMP',
                     'VID_END_FRAME', 'VID_END_ELV', 'VID_END_LAT', 'VID_END_LNG', 'VID_END_TSTMP',
                     'SCN_END_FRAME', 'SCN_END_ELV', 'SCN_END_LAT', 'SCN_END_LNG', 'SCN_END_TSTMP']

# Iterate over the rows in the augmented dataframe
for i, _ in augmented_df.iterrows():
    # Select a random row from df_with_json
    random_index = random.randint(0, len(df_with_json) - 1)
    random_row = df_with_json.iloc[random_index]
    
    # Assign values from the random row to the augmented dataframe
    for column in df_with_json.columns:
        if column in numerical_columns:
            augmented_df.at[i, column] = random_row[column]

In [ ]:
augmented_df

In [ ]:
augmented_df["VID_START_TSTMP"]

In [ ]:
augmented_df["SCN_START_TSTMP"]

In [ ]:
# Iterate over the rows in the augmented dataframe
for i, row in augmented_df.iterrows():
    if pd.isnull(row['VID_START_TSTMP']):
        continue
    
    # Randomly add or subtract 8 seconds to 'VID_START_TSTMP'
    vid_start_timestamp = row['VID_START_TSTMP'] + random.choice([-1, 1]) * 8 * 1e9
    
    # Randomly add or subtract 8 seconds to 'SCN_START_TSTMP'
    scn_start_timestamp = row['SCN_START_TSTMP'] + random.choice([-1, 1]) * 8 * 1e9
    
    # Update the values in the augmented dataframe
    augmented_df.at[i, 'VID_START_TSTMP'] = vid_start_timestamp
    augmented_df.at[i, 'SCN_START_TSTMP'] = scn_start_timestamp


In [ ]:
augmented_df

In [ ]:
augmented_df["VID_START_TSTMP"]

In [ ]:
augmented_df["SCN_START_TSTMP"]

In [ ]:
# Iterate over the rows in the augmented dataframe
for i, row in augmented_df.iterrows():
    if pd.isnull(row['SCN_END_TSTMP']) or pd.isnull(row['VID_END_TSTMP']):
        continue
    
    # Randomly add or subtract 3 seconds to 'SCN_END_TSTMP'
    scn_end_timestamp = row['SCN_END_TSTMP'] + random.choice([-1, 1]) * 3 * 1e9
    
    # Randomly add or subtract 3 seconds to 'VID_END_TSTMP'
    vid_end_timestamp = row['VID_END_TSTMP'] + random.choice([-1, 1]) * 3 * 1e9
    
    # Update the values in the augmented dataframe
    augmented_df.at[i, 'SCN_END_TSTMP'] = scn_end_timestamp
    augmented_df.at[i, 'VID_END_TSTMP'] = vid_end_timestamp

In [ ]:
augmented_df

In [ ]:
# Concatenate the original dataframe with the augmented dataframe
combined_df = pd.concat([df_with_json, augmented_df], ignore_index=True)

In [ ]:
combined_df

In [ ]:
# Set the initial value of the VALIDITY_SCN_START column as "Invalid"
combined_df['VALIDITY_SCN_START'] = 'Invalid'

# Update the VALIDITY_SCN_START column based on the conditions
combined_df.loc[(combined_df['VID_START_TSTMP'].notnull()) & (combined_df['SCN_START_TSTMP'].notnull()), 'VALIDITY_SCN_START'] = \
    combined_df.loc[(combined_df['VID_START_TSTMP'].notnull()) & (combined_df['SCN_START_TSTMP'].notnull())].apply(
        lambda row: 'Valid' if (row['SCN_START_TSTMP'] - row['VID_START_TSTMP']) < 30 * 1e9 else 'Invalid', axis=1)

# Set the initial value of the VALIDITY_SCN_START column as "Invalid"
combined_df['VALIDITY_SCN_END'] = 'Invalid'

# Update the VALIDITY_SCN_START column based on the conditions
combined_df.loc[(combined_df['VID_END_TSTMP'].notnull()) & (combined_df['SCN_END_TSTMP'].notnull()), 'VALIDITY_SCN_END'] = \
    combined_df.loc[(combined_df['VID_END_TSTMP'].notnull()) & (combined_df['SCN_END_TSTMP'].notnull())].apply(
        lambda row: 'Valid' if (row['VID_END_TSTMP'] - row['SCN_END_TSTMP']) < 10 * 1e9 else 'Invalid', axis=1)

In [ ]:
# Count the occurrences of each validity value
validity_counts_start = combined_df['VALIDITY_SCN_START'].value_counts()
validity_counts_end = combined_df['VALIDITY_SCN_END'].value_counts()

# Create separate bar plots for Scene Start Validity and Scene End Validity
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6))

# Plot for Scene Start Validity
ax1.bar(validity_counts_start.index, validity_counts_start.values, color='blue', alpha=0.5)
ax1.set_xlabel('Validity')
ax1.set_ylabel('Count')
ax1.set_title('Validity of Scene Start')

# Plot for Scene End Validity
ax2.bar(validity_counts_end.index, validity_counts_end.values, color='green', alpha=0.5)
ax2.set_xlabel('Validity')
ax2.set_ylabel('Count')
ax2.set_title('Validity of Scene End')

# Adjust spacing between subplots
plt.tight_layout()

# Show the plots
plt.show()

In [ ]:
# Create the CLIP_VALIDITY column
combined_df['CLIP_VALIDITY'] = np.where(
    (combined_df['VALIDITY_SCN_START'] == 'Invalid') | (combined_df['VALIDITY_SCN_END'] == 'Invalid'),
    'Invalid',
    'Valid'
)

In [ ]:
# Count the occurrences of each value in the CLIP_VALIDITY column
validity_counts = combined_df['CLIP_VALIDITY'].value_counts()

# Create a bar plot
plt.bar(validity_counts.index, validity_counts.values)

# Set the plot title and labels
plt.title('Clip Validity')
plt.xlabel('Validity')
plt.ylabel('Count')

# Show the plot
plt.show()

In [ ]:
combined_df

In [ ]:
combined_df.to_csv('flask_data_no_encoder.csv', index=False)

In [ ]:
# List of non-numeric columns (excluding 'Clip')
non_numeric_columns = [col for col in combined_df.columns if col != 'Clip' and not combined_df[col].dtype.kind in 'biufc']

# Plotting options
fig, axs = plt.subplots(len(non_numeric_columns), 1, figsize=(10, len(non_numeric_columns) * 5))

# Iterate over each categorical column
for i, column in enumerate(non_numeric_columns):
    # Get unique values and their counts
    values = combined_df[column].value_counts()

    # Create a bar plot
    axs[i].bar(values.index, values.values)

    # Set plot title and labels
    axs[i].set_title(f'Column: {column}')
    axs[i].set_xlabel('Categories')
    axs[i].set_ylabel('Count')
#Adjust spacing between subplots
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
# Create a new dataframe with the categorical columns from the combined dataframe
combined_categorical = combined_df[non_numeric_columns]

# Initialize the LabelEncoder
label_encoder = LabelEncoder()

label_mappings = {}
# Iterate over each categorical column and apply label encoding
for column in combined_categorical:
# Extract and store the label encoder mappings for each column

    combined_df[column] = label_encoder.fit_transform(combined_df[column])
    label_mappings[column] = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

# Print the label encoder mappings for each column
print("Label Encoder Mappings:")
for column, mappings in label_mappings.items():
    print(f"{column}: {mappings}")

In [ ]:
for column in combined_categorical:
    print(column)

In [ ]:
combined_df

In [ ]:
combined_df = combined_df.drop("DrivingMode", axis=1)

In [ ]:
combined_df

In [ ]:
first_row = combined_df.iloc[0].values.tolist()
print(first_row)

In [ ]:
combined_df.columns

In [ ]:
combined_df.to_csv('flask_data.csv', index=False)


# Training & Testing

In [ ]:
X = combined_df.fillna(combined_df.mean()).drop(["Clip", "CLIP_VALIDITY", 'VALIDITY_SCN_START', 'VALIDITY_SCN_END'], axis=1)
y = combined_df["CLIP_VALIDITY"]

In [ ]:
X.columns

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
classifiers = [
    ("xgBoost", XGBClassifier()),
    ("AdaBoost", AdaBoostClassifier()),
    ("GradientBoost", GradientBoostingClassifier()),
    ("Logistic Regression", LogisticRegression()),
    ("Random Forest", RandomForestClassifier()),
    ("Random Forest with AdaBoost", RandomForestClassifier(n_estimators=100)),
    ("KNN", KNeighborsClassifier()),
    ("Decision Tree", DecisionTreeClassifier()),
    ("SVM", SVC()),
    ("Naive Bayes", GaussianNB())
]

In [ ]:
for name, classifier in classifiers:
    classifier.fit(X_train, y_train)
    y_pred = classifier.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    print(f"Classifier: {name}")
    print(f"Accuracy: {accuracy}")
    print(f"Precision: {precision}")
    print(f"Recall: {recall}")
    print(f"F1-score: {f1}")
    # Calculate the confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    # Create a heatmap of the confusion matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, cmap="Blues", fmt="d", cbar=False)
    plt.title(f"Confusion Matrix for {name}")
    plt.xlabel("Predicted Labels")
    plt.ylabel("True Labels")
    plt.show()
    print("----------------------")

# Evaluation

In [ ]:
# Initialize classifiers
classifiers = [
    XGBClassifier(),
    AdaBoostClassifier(),
    GradientBoostingClassifier(),
    LogisticRegression(),
    RandomForestClassifier(),
    RandomForestClassifier(n_estimators=100),
    KNeighborsClassifier(),
    DecisionTreeClassifier(),
    SVC(probability=True),  # Enable probability estimation for SVC
    GaussianNB()
]

# Plot ROC curve for each classifier
plt.figure(figsize=(10, 8))
for classifier in classifiers:
    # Train the classifier
    classifier.fit(X_train, y_train)
    
    # Make predictions on the testing set
    if hasattr(classifier, "predict_proba"):
        # Use predict_proba for classifiers that support it
        y_pred_prob = classifier.predict_proba(X_test)[:, 1]
    else:
        # Use decision_function for classifiers that don't support predict_proba
        y_pred_prob = classifier.decision_function(X_test)
    
    # Compute the false positive rate, true positive rate, and thresholds
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
    
    # Calculate the area under the ROC curve
    roc_auc = roc_auc_score(y_test, y_pred_prob)
    
    # Plot the ROC curve
    plt.plot(fpr, tpr, label=f'{classifier.__class__.__name__} (AUC = {roc_auc:.2f})')

# Set plot labels and title
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend()

# Show the plot
plt.show()


In [ ]:
# Train and save the XGBoost classifier
xgb_classifier = XGBClassifier()
xgb_classifier.fit(X_train, y_train)
model_file_xgb = 'model_xgBoost.pkl'
joblib.dump(xgb_classifier, model_file_xgb)
print(f'Saved XGBoost model to {model_file_xgb}')

# Train and save the AdaBoost classifier
adaboost_classifier = AdaBoostClassifier()
adaboost_classifier.fit(X_train, y_train)
model_file_adaboost = 'model_AdaBoost.pkl'
joblib.dump(adaboost_classifier, model_file_adaboost)
print(f'Saved AdaBoost model to {model_file_adaboost}')

# Train and save the GradientBoosting classifier
gradientboost_classifier = GradientBoostingClassifier()
gradientboost_classifier.fit(X_train, y_train)
model_file_gradientboost = 'model_GradientBoost.pkl'
joblib.dump(gradientboost_classifier, model_file_gradientboost)
print(f'Saved GradientBoosting model to {model_file_gradientboost}')

# Train and save the Random Forest classifier
randomforest_classifier = RandomForestClassifier()
randomforest_classifier.fit(X_train, y_train)
model_file_randomforest = 'model_RandomForest.pkl'
joblib.dump(randomforest_classifier, model_file_randomforest)
print(f'Saved Random Forest model to {model_file_randomforest}')

# Train and save the Logistic Regression classifier
logistic_regression_classifier = LogisticRegression()
logistic_regression_classifier.fit(X_train, y_train)
model_file_logisticregression = 'model_LogisticRegression.pkl'
joblib.dump(logistic_regression_classifier, model_file_logisticregression)
print(f'Saved Logistic Regression model to {model_file_logisticregression}')

# Train and save the KNN classifier
knn_classifier = KNeighborsClassifier()
knn_classifier.fit(X_train, y_train)
model_file_knn = 'model_KNN.pkl'
joblib.dump(knn_classifier, model_file_knn)
print(f'Saved KNN model to {model_file_knn}')

# Train and save the Decision Tree classifier
decisiontree_classifier = DecisionTreeClassifier()
decisiontree_classifier.fit(X_train, y_train)
model_file_decisiontree = 'model_DecisionTree.pkl'
joblib.dump(decisiontree_classifier, model_file_decisiontree)
print(f'Saved Decision Tree model to {model_file_decisiontree}')

# Train and save the SVM classifier
svm_classifier = SVC()
svm_classifier.fit(X_train, y_train)
model_file_svm = 'model_SVM.pkl'
joblib.dump(svm_classifier, model_file_svm)
print(f'Saved SVM model to {model_file_svm}')